# Trustworthy Personalised AI — Analysis Dashboard

Research notebook for analysing training data quality, model benchmark results, and conversation samples. All charts are saved to `exports/` as SVG (vector, dissertation-ready) and PNG (high-resolution raster) via plotly + kaleido. Run cells top-to-bottom on first use; individual sections can be re-run independently after that.

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import HTML, display

pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Set2

DATA_DIR    = Path("data")
REPORTS_DIR = Path("reports")
EXPORTS_DIR = Path("exports")
EXPORTS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    """Save dissertation-ready SVG + high-res PNG and show inline."""
    fig.write_image(str(EXPORTS_DIR / f"{name}.svg"))
    fig.write_image(str(EXPORTS_DIR / f"{name}.png"), scale=3)
    print(f"\u2713 exports/{name}.svg + .png")
    fig.show()

## Section 1 — Data Loading

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def extract_tag_len(content, tag):
    """Return total character length of ALL <tag>\u2026</tag> blocks, else 0."""
    return sum(len(m.strip()) for m in re.findall(rf"<{tag}>(.*?)</{tag}>", content, re.DOTALL))

SPLITS = {
    "train":             DATA_DIR / "train_sft_v2.jsonl",
    "eval":              DATA_DIR / "eval_sft_v2.jsonl",
    "train_interleaved": DATA_DIR / "train_interleaved.jsonl",
    "train_partB":       DATA_DIR / "train_partB.jsonl",
}

ALL_RECORDS = []   # full records — used by conversation renderer (Section 7)
rows = []

for split_name, path in SPLITS.items():
    if not path.exists():
        continue
    for rec in load_jsonl(path):
        meta = rec.get("metadata", {}).copy()
        msgs = rec.get("messages", [])
        asst = " ".join(m["content"] for m in msgs if m["role"] == "assistant")
        rows.append({
            **meta,
            "split":          split_name,
            "num_messages":   len(msgs),
            "response_chars": len(asst),
            "think_chars":    extract_tag_len(asst, "think"),
            "answer_chars":   extract_tag_len(asst, "answer"),
            "_idx":           len(ALL_RECORDS),
        })
        ALL_RECORDS.append(rec)

df = pd.DataFrame(rows)
df["category"]     = df["category"].fillna("(none)")
df["tool_profile"] = df["tool_profile"].fillna("(none)")
print(f"Loaded {len(df):,} records | {df['split'].nunique()} splits")
print(df.groupby("split").size().to_string())

## Section 2 — Dataset Overview

In [ ]:
# --- 2a: Summary metric cards ---
total      = len(df)
avg_score  = df["constitution_score"].dropna().mean() if "constitution_score" in df.columns else float("nan")
avg_len    = df["response_chars"].mean()
rev_pct    = df["revised"].dropna().mean() * 100 if "revised" in df.columns else float("nan")

metrics = [
    (f"{total:,}",            "Total Records"),
    (f"{df['split'].nunique()}", "Splits"),
    (f"{df['category'].nunique()}", "Categories"),
    (f"{avg_score:.3f}",      "Avg Score (train+eval)"),
    (f"{avg_len:,.0f}",       "Avg Response Length (chars)"),
    (f"{rev_pct:.0f}%",       "Revised (train+eval)"),
]
cards = "".join(f"""
  <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;
              padding:16px 24px;min-width:140px;text-align:center">
    <div style="font-size:26px;font-weight:700;color:#1e293b">{v}</div>
    <div style="font-size:12px;color:#64748b;margin-top:4px">{label}</div>
  </div>""" for v, label in metrics)
display(HTML(f"""
<div style="display:flex;gap:16px;flex-wrap:wrap;font-family:ui-sans-serif,sans-serif;
            margin:12px 0">{cards}</div>"""))

In [ ]:
# --- 2b: Category distribution donut ---
cat_counts = df["category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]
fig = px.pie(
    cat_counts, names="category", values="count",
    title="Training Data — Category Distribution",
    hole=0.45, color_discrete_sequence=PALETTE
)
fig.update_traces(textposition="outside", textinfo="label+percent")
fig.update_layout(showlegend=False, margin=dict(t=50, b=20))
save_fig(fig, "01_category_distribution")

In [ ]:
# --- 2c: Records per category per split (grouped bar) ---
split_cat = df.groupby(["split", "category"]).size().reset_index(name="count")
fig = px.bar(
    split_cat, x="category", y="count", color="split",
    barmode="group",
    title="Records per Category by Split",
    color_discrete_sequence=PALETTE,
    labels={"count": "Count", "category": "Category", "split": "Split"}
)
fig.update_xaxes(tickangle=-30)
save_fig(fig, "02_split_category_counts")

## Section 3 — Constitution Quality Analysis

In [ ]:
# --- 3a: Score distribution histogram ---
fig = px.histogram(
    df, x="constitution_score", nbins=20,
    title="Constitution Score Distribution",
    labels={"constitution_score": "Score (0–1)", "count": "Records"},
    color_discrete_sequence=[PALETTE[0]]
)
fig.update_layout(bargap=0.05)
save_fig(fig, "03_score_distribution")

In [ ]:
# --- 3b: Score by category ---
score_df = df[df["constitution_score"].notna()].copy()
fig = px.box(
    score_df, x="category", y="constitution_score",
    color="category", color_discrete_sequence=PALETTE,
    title="Constitution Score by Category",
    labels={"constitution_score": "Score", "category": "Category"},
    points="all"
)
fig.update_layout(showlegend=False)
fig.update_xaxes(tickangle=-30)
save_fig(fig, "04_score_by_category")

In [ ]:
# --- 3c: Draft violations histogram ---
viol_df = df[df["constitution_violations_in_draft"].notna()].copy()
fig = px.histogram(
    viol_df, x="constitution_violations_in_draft",
    title="Constitution Violations in Draft",
    labels={"constitution_violations_in_draft": "Violations", "count": "Records"},
    color_discrete_sequence=[PALETTE[1]]
)
fig.update_layout(bargap=0.1)
save_fig(fig, "05_violations_histogram")

In [ ]:
# --- 3d: Score vs violations scatter ---
sv_df = df[df["constitution_score"].notna() & df["constitution_violations_in_draft"].notna()].copy()
fig = px.scatter(
    sv_df, x="constitution_violations_in_draft", y="constitution_score",
    color="category",
    title="Constitution Score vs. Violations in Draft",
    labels={"constitution_violations_in_draft": "Violations in Draft",
            "constitution_score": "Final Score"},
    color_discrete_sequence=PALETTE,
    hover_data=["category", "tool_profile"]
)
save_fig(fig, "06_score_vs_violations")